# Backtesting del modelo (capa de Momentum) — datos gratis (Yahoo)

Sigue tu plan: datos históricos → definir éxito (ganarle al S&P 500) → walk-forward
→ optimizar pesos → **validar fuera de muestra**.

## ⚠️ LÍMITES (léelos, son la diferencia entre honesto y engañoso)
- **Solo se backtestea el MOMENTUM.** Es lo único de tu modelo que se puede calcular
  sin trampa con Yahoo (usa solo precios pasados). Los fundamentales NO se pueden
  backtestear con datos gratis (necesitarían los ratios como se veían en cada fecha
  del pasado; Yahoo solo da los de hoy → sería hacer trampa con datos del futuro).
- **Survivorship bias:** el universo son acciones que SIGUEN vivas. Las que quebraron
  no están → los resultados salen optimistas.
- **~9 años y un solo universo** = poca potencia estadística. Un resultado, bueno o
  malo, no es concluyente.
- El **TEST (fuera de muestra)** es el juez honesto. Si el exceso desaparece en TEST,
  la señal es ruido, por bonito que se vea el TRAIN.
- NO es asesoría financiera.

Corre las celdas en orden.

### 1. Instalar

In [ ]:
!pip install yfinance scipy -q

### 2. Motor del backtest (no edites)

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import minimize

FEATS = ["ret_1m","ret_3m","ret_6m","ret_12m","dist_sma50","dist_sma200","mom_vol"]
DIRS  = {"ret_1m":1,"ret_3m":1,"ret_6m":1,"ret_12m":1,"dist_sma50":1,"dist_sma200":1,"mom_vol":-1}

def build_features(prices):
    """prices: DataFrame (fechas x tickers). Devuelve dict feat -> DataFrame point-in-time."""
    f = {}
    f["ret_1m"]  = prices/prices.shift(21)-1
    f["ret_3m"]  = prices/prices.shift(63)-1
    f["ret_6m"]  = prices/prices.shift(126)-1
    f["ret_12m"] = prices/prices.shift(252)-1
    f["dist_sma50"]  = prices/prices.rolling(50).mean()-1
    f["dist_sma200"] = prices/prices.rolling(200).mean()-1
    r = prices.pct_change()
    f["mom_vol"] = r.rolling(63).std()*np.sqrt(252)
    return f

def score_at(feats, i, pesos):
    """Score cross-seccional en la fila i usando SOLO datos hasta i (point-in-time)."""
    s = pd.Series(0.0, index=feats["ret_1m"].columns); den=0.0
    for k in FEATS:
        row = feats[k].iloc[i]
        if row.notna().sum() < 3: continue
        r = row.rank(pct=True)*100
        pct = r if DIRS[k]>0 else 100-r
        s = s + pct.fillna(50)*pesos[k]; den += pesos[k]
    return s/den if den>0 else s

def walk_forward(prices, spx, pesos, topn=10, rebal=21, start=252):
    """Backtest walk-forward: cada 'rebal' dias, elige top-N por score y mide el
    retorno hasta el siguiente rebalanceo. Sin look-ahead."""
    feats = build_features(prices)
    idxs = list(range(start, len(prices)-rebal, rebal))
    ret_strat, ret_bench, fechas = [], [], []
    hit12 = []
    for i in idxs:
        sc = score_at(feats, i, pesos)
        top = sc.dropna().sort_values(ascending=False).head(topn).index
        if len(top)==0: continue
        p0 = prices.iloc[i]; p1 = prices.iloc[i+rebal]
        r_stock = (p1[top]/p0[top]-1).mean()
        r_bench = spx.iloc[i+rebal]/spx.iloc[i]-1
        ret_strat.append(r_stock); ret_bench.append(r_bench); fechas.append(prices.index[i])
        # exito a 12m: portafolio vs S&P en 252 dias
        if i+252 < len(prices):
            r12_s = (prices.iloc[i+252][top]/p0[top]-1).mean()
            r12_b = spx.iloc[i+252]/spx.iloc[i]-1
            hit12.append(1 if r12_s>r12_b else 0)
    eq_s = pd.Series(np.cumprod([1+x for x in ret_strat]), index=fechas)
    eq_b = pd.Series(np.cumprod([1+x for x in ret_bench]), index=fechas)
    return {"ret_strat":np.array(ret_strat),"ret_bench":np.array(ret_bench),
            "eq_s":eq_s,"eq_b":eq_b,"hit12":np.mean(hit12) if hit12 else np.nan,
            "n_rebal":len(ret_strat)}

def metricas(res, periodos_por_anio=12):
    rs, rb = res["ret_strat"], res["ret_bench"]
    ann = lambda r: (np.prod(1+r))**(periodos_por_anio/len(r))-1
    vol = lambda r: r.std()*np.sqrt(periodos_por_anio)
    dd = lambda eq: (eq/eq.cummax()-1).min()
    return {"cagr_s":ann(rs),"cagr_b":ann(rb),"exceso":ann(rs)-ann(rb),
            "vol_s":vol(rs),"sharpe_s":(ann(rs)-0.04)/vol(rs) if vol(rs)>0 else np.nan,
            "sharpe_b":(ann(rb)-0.04)/vol(rb) if vol(rb)>0 else np.nan,
            "mdd_s":dd(res["eq_s"]),"mdd_b":dd(res["eq_b"]),
            "hit12":res["hit12"],"n":res["n_rebal"]}

def preparar(prices, spx, topn=10, rebal=21, start=252):
    """Precalcula percentiles y retornos futuros en cada rebalanceo (una sola vez)."""
    feats=build_features(prices); idxs=list(range(start,len(prices)-rebal,rebal))
    pasos=[]
    for i in idxs:
        pm={}
        for k in FEATS:
            row=feats[k].iloc[i]; r=row.rank(pct=True)*100
            pm[k]=(r if DIRS[k]>0 else 100-r).fillna(50)
        pmat=pd.DataFrame(pm)
        fwd1=(prices.iloc[i+rebal]/prices.iloc[i]-1)
        b1=spx.iloc[i+rebal]/spx.iloc[i]-1
        pasos.append({"pmat":pmat,"fwd1":fwd1,"b1":b1,"valid":prices.iloc[i].notna()})
    return pasos, topn

def evaluar(pasos, topn, pesos):
    rs, rb=[], []
    w=np.array([pesos[k] for k in FEATS]); w=w/w.sum() if w.sum()>0 else w
    for p in pasos:
        sc=(p["pmat"][FEATS].values@w)
        sc=pd.Series(sc,index=p["pmat"].index).where(p["valid"])
        top=sc.dropna().sort_values(ascending=False).head(topn).index
        if len(top)==0: continue
        rs.append(p["fwd1"][top].mean()); rb.append(p["b1"])
    rs, rb=np.array(rs),np.array(rb)
    ann=lambda r:(np.prod(1+r))**(12/len(r))-1 if len(r) else np.nan
    return ann(rs)-ann(rb), rs, rb

def optimizar_oos(prices, spx, topn=10):
    pasos,_=preparar(prices,spx,topn)
    n=len(pasos); corte=int(n*0.7)
    train, test=pasos[:corte], pasos[corte:]
    # pesos iguales (baseline)
    base=dict.fromkeys(FEATS,1.0)
    ex_tr_base,_,_=evaluar(train,topn,base); ex_te_base,_,_=evaluar(test,topn,base)
    # optimizar en TRAIN
    def neg(wv):
        w=np.clip(wv,0,None); pesos=dict(zip(FEATS,w))
        ex,_,_=evaluar(train,topn,pesos); return -ex
    res=minimize(neg,np.ones(len(FEATS)),method="Nelder-Mead",
                 options={"maxiter":300,"xatol":1e-3,"fatol":1e-4})
    w=np.clip(res.x,0,None); pesos_opt=dict(zip(FEATS,w))
    ex_tr_opt,_,_=evaluar(train,topn,pesos_opt); ex_te_opt,_,_=evaluar(test,topn,pesos_opt)
    return {"base_train":ex_tr_base,"base_test":ex_te_base,
            "opt_train":ex_tr_opt,"opt_test":ex_te_opt,
            "pesos_opt":{k:round(v,2) for k,v in zip(FEATS,w/ w.sum())}}

### 3. Descargar datos (universo + S&P 500, 10 años)

In [ ]:
import yfinance as yf
UNIVERSO = ["AAPL","MSFT","NVDA","AVGO","ORCL","CRM","ADBE","CSCO","AMD","QCOM",
 "GOOGL","META","NFLX","DIS","TMUS","VZ","AMZN","TSLA","HD","MCD","NKE","SBUX","LOW",
 "PG","KO","PEP","COST","WMT","PM","MDLZ","JPM","BAC","WFC","GS","V","MA","AXP",
 "LLY","UNH","JNJ","MRK","ABBV","PFE","TMO","CAT","GE","BA","HON","UPS","RTX","DE",
 "XOM","CVX","COP","SLB","EOG","LIN","SHW","FCX","NEM","APD","NEE","DUK","SO","D","AEP",
 "PLD","AMT","EQIX","SPG","O"]
data = yf.download(UNIVERSO+["^GSPC"], period="10y", auto_adjust=True, progress=False)["Close"]
data = data.dropna(how="all").ffill()
spx = data["^GSPC"].dropna()
prices = data[[c for c in UNIVERSO if c in data.columns]].dropna(how="all")
prices, spx = prices.align(spx, join="inner", axis=0)
print(f"Datos: {prices.shape[1]} acciones, {len(prices)} dias ({len(prices)/252:.1f} años)")

### 4. Backtest + optimización + validación fuera de muestra

In [ ]:
import matplotlib.pyplot as plt

# Backtest con pesos iguales (baseline) sobre todo el periodo
pesos_iguales = dict.fromkeys(FEATS, 1.0)
res = walk_forward(prices, spx, pesos_iguales, topn=10)
m = metricas(res)

print("="*56)
print("  BACKTEST — Momentum, top 10, rebalanceo mensual")
print("="*56)
print(f"Rebalanceos: {m['n']}  |  ~{m['n']/12:.1f} años probados")
print(f"\nCAGR estrategia:  {m['cagr_s']*100:+.1f}%")
print(f"CAGR S&P 500:     {m['cagr_b']*100:+.1f}%")
print(f"EXCESO anual:     {m['exceso']*100:+.1f}%   <-- lo que importa")
print(f"\nSharpe estrategia {m['sharpe_s']:.2f}  vs  S&P {m['sharpe_b']:.2f}")
print(f"Peor caida estrat {m['mdd_s']*100:.0f}%  vs  S&P {m['mdd_b']*100:.0f}%")
print(f"Gana al S&P a 12 meses: {m['hit12']*100:.0f}% de las veces")
print("="*56)

# Optimizacion + fuera de muestra
o = optimizar_oos(prices, spx, topn=10)
print("\nVALIDACION FUERA DE MUESTRA (70% train / 30% test):")
print("-"*56)
print(f"  Pesos IGUALES  -> exceso TRAIN {o['base_train']*100:+.1f}%   TEST {o['base_test']*100:+.1f}%")
print(f"  Pesos OPTIMIZADOS -> exceso TRAIN {o['opt_train']*100:+.1f}%   TEST {o['opt_test']*100:+.1f}%")
print("-"*56)
if o['opt_test'] < o['base_test'] or o['opt_test'] < o['opt_train']*0.5:
    print("VEREDICTO: optimizar NO mejoro fuera de muestra -> los pesos 'optimos'")
    print("estaban sobreajustados al pasado (overfitting). La señal simple es mas honesta.")
else:
    print("VEREDICTO: el exceso sobrevive fuera de muestra -> señal mas robusta (con cautela).")
print(f"\nPesos optimizados (normalizados): {o['pesos_opt']}")

# Grafica: crecimiento de $1
plt.figure(figsize=(9,4))
plt.plot(res["eq_s"].index, res["eq_s"].values, label="Estrategia momentum", color="#2c6fbb", lw=1.6)
plt.plot(res["eq_b"].index, res["eq_b"].values, label="S&P 500", color="#7f8c8d", lw=1.4, ls="--")
plt.title("Crecimiento de $1 — estrategia vs S&P 500 (con survivorship bias)")
plt.legend(); plt.grid(alpha=.3); plt.show()

print("\nRECORDATORIO: survivorship bias + solo momentum + ~9 años. Resultado")
print("orientativo, NO prueba definitiva. El TEST manda sobre el TRAIN.")